# 02. 실습: Centering과 Joint Conditioning

목표: OPD²의 핵심 안정화 장치인 centering과 joint conditioning을 구현합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 분포와 reward 함수 준비

이전 노트북과 같은 toy distribution을 사용합니다. Student 확률은 expectation을 계산하는 기준이 됩니다.

In [ ]:
import math

student = {"therefore": 0.08, "perhaps": 0.22, "multiply": 0.30, "add": 0.18, "answer": 0.22}
teacher_base = {"therefore": 0.10, "perhaps": 0.25, "multiply": 0.28, "add": 0.12, "answer": 0.25}
teacher = {"therefore": 0.24, "perhaps": 0.08, "multiply": 0.10, "add": 0.38, "answer": 0.20}


def logp(distribution, token):
    return math.log(distribution[token])


def opd_reward(token):
    return logp(teacher, token) - logp(student, token)


def delta_reward(token):
    return logp(teacher, token) - logp(teacher_base, token)


def expected_reward(reward_fn, policy):
    return sum(policy[token] * reward_fn(token) for token in policy)


expected_opd = expected_reward(opd_reward, student)
expected_delta = expected_reward(delta_reward, student)
print("E[R_OPD]   =", round(expected_opd, 3))
print("E[R_delta] =", round(expected_delta, 3))

## 2. Centered advantage 계산

Centering은 reward에서 평균을 빼서 token별 상대적 방향을 더 분명하게 만듭니다.

In [ ]:
def centered_advantage(reward_fn, token, policy):
    return reward_fn(token) - expected_reward(reward_fn, policy)


def opd_advantage(token):
    return centered_advantage(opd_reward, token, student)


def delta_advantage(token):
    return centered_advantage(delta_reward, token, student)


print("token | A_OPD | A_delta")
print("--- | --- | ---")
for token in student:
    print(f"{token:9s} | {opd_advantage(token):7.3f} | {delta_advantage(token):8.3f}")

## 3. Joint conditioning 적용

OPD²는 delta advantage와 OPD advantage가 같은 부호일 때만 delta update를 적용합니다. 방향이 충돌하면 0으로 둡니다.

In [ ]:
def opd2_advantage(token):
    a_delta = delta_advantage(token)
    a_opd = opd_advantage(token)
    return a_delta if a_delta * a_opd > 0 else 0.0


print("token | A_OPD2 | update?")
print("--- | --- | ---")
for token in student:
    advantage = opd2_advantage(token)
    print(f"{token:9s} | {advantage:8.3f} | {advantage != 0.0}")

## 4. Toy policy update

실제 모델은 gradient로 log probability를 업데이트합니다. 여기서는 직관을 위해 token score에 advantage를 더하고 softmax로 새 분포를 만듭니다.

In [ ]:
def softmax(scores):
    max_score = max(scores.values())
    exp_scores = {token: math.exp(score - max_score) for token, score in scores.items()}
    total = sum(exp_scores.values())
    return {token: value / total for token, value in exp_scores.items()}


def update_policy(policy, advantage_fn, learning_rate=0.4):
    scores = {token: math.log(prob) + learning_rate * advantage_fn(token) for token, prob in policy.items()}
    return softmax(scores)


updated = update_policy(student, opd2_advantage)
print("token | before | after")
print("--- | --- | ---")
for token in student:
    print(f"{token:9s} | {student[token]:6.3f} | {updated[token]:6.3f}")